# Recursión

En este notebook vamos a ver un ejemplo de recursión en SQL. Para esto vamos a usar psycopg2 para conectarnos a PostgreSQL.

In [1]:
import psycopg2

# Modifica los parámetros de conexión según tu configuración local
conn = psycopg2.connect(
    host="localhost",
    dbname="progsql",
    user="tu_usuario",
    password=""
)
conn.autocommit = True
cursor = conn.cursor()

In [6]:
def print_rows(cursor):
    rows = cursor.fetchall()
    if not rows:
        print("(0 rows)")
        return rows
    cols = [d[0] for d in cursor.description]
    widths = [max(len(str(x)) for x in col) for col in zip(cols, *rows)]
    header = " | ".join(f"{c:<{w}}" for c, w in zip(cols, widths))
    print(header)
    print("-" * len(header))
    for row in rows:
        print(" | ".join(f"{str(v):<{w}}" for v, w in zip(row, widths)))
    #return rows

In [3]:
cursor.execute("""CREATE TABLE Vuelos(ciudad_origen VARCHAR(100), ciudad_destino VARCHAR(100),
               PRIMARY KEY(ciudad_origen, ciudad_destino))""")

In [4]:
cursor.execute("INSERT INTO Vuelos VALUES('Santiago', 'Buenos Aires')")
cursor.execute("INSERT INTO Vuelos VALUES('Buenos Aires', 'Madrid')")
cursor.execute("INSERT INTO Vuelos VALUES('Madrid', 'Doha')")
cursor.execute("INSERT INTO Vuelos VALUES('Doha', 'Katmandú')")
cursor.execute("INSERT INTO Vuelos VALUES('Santiago', 'Lima')")
cursor.execute("INSERT INTO Vuelos VALUES('Lima', 'Miami')")

In [ ]:
cursor.execute("""
WITH RECURSIVE Alcanzo(ciudad_origen, ciudad_destino) AS (
    SELECT * FROM Vuelos
    UNION
    SELECT V.ciudad_origen, A.ciudad_destino
    FROM Vuelos V, Alcanzo A
    WHERE V.ciudad_destino = A.ciudad_origen
)
SELECT * FROM Alcanzo WHERE ciudad_origen='Santiago'
""")
print_rows(cursor)